# 01 · Ingestion

Loads the raw YouTube Data API responses (`data/raw/*.json`, one file per video) into Python and builds one flat table with a row per comment.

The assignment asks for four structures, each with its own section below:

| Structure | Used for |
|---|---|
| **list** | the raw files as loaded, and the flattened comment records |
| **dict** | lookups: video metadata by `video_id`, commenter country by channel ID, dedup by `comment_id` |
| **pandas Series** | a single column: comment text, and comment counts per video |
| **pandas DataFrame** | the full comment table that every later notebook uses |

**Output:** `data/comments_raw.csv`, the raw dataset deliverable (one row per comment, before cleaning).

**Privacy:** commenter display names and profile URLs are not copied into the table. Commenters are identified only by their channel ID, which is enough to count unique commenters and join the country field.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import config

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 120)
import json
from collections import Counter

## 1. Lists: load the raw files

Each raw file holds the video's metadata, the unmodified `commentThreads.list` pages, and the `comments.list` reply pages for threads with more than 5 replies.

In [2]:
raw_files = []   # list of dicts, one per video
for path in sorted(config.RAW_DIR.glob("*.json")):
    with open(path, encoding="utf-8") as f:
        raw_files.append(json.load(f))

print(f"{len(raw_files)} raw files loaded")
print("top-level keys:", list(raw_files[0].keys()))

18 raw files loaded
top-level keys: ['video', 'collected_at_utc', 'comment_threads_pages', 'reply_pages']


## 2. Dictionaries: lookup tables

In [3]:
# video_id -> video metadata (creator, role, period, topic pair, publish time, title, ...)
# Taken from data/videos.csv, the curated list. The copy saved inside each raw file
# reflects the list at collection time, and early files predate the `role` column.
videos = pd.read_csv(config.VIDEOS_CSV).set_index("video_id", drop=False).to_dict(orient="index")
assert set(videos) == {d["video"]["video_id"] for d in raw_files}, "videos.csv and data/raw do not match"

# commenter channel ID -> self-reported country (most are blank)
with open(config.DATA_DIR / "commenter_channels.json", encoding="utf-8") as f:
    channel_pages = json.load(f)["pages"]
country_by_channel = {
    item["id"]: item["snippet"].get("country")
    for page in channel_pages
    for item in page.get("items", [])
}

print(f"{len(videos)} videos, {len(country_by_channel)} commenter channels")
videos[next(iter(videos))]

18 videos, 4870 commenter channels


{'role': 'primary',
 'creator': 'pate',
 'video_id': 'karqas42RcE',
 'period': 'before',
 'topic_pair': '1_cfp_rankings_pre',
 'published_utc': '2026-11-04T02:17:46Z',
 'hours_vs_announce': -133.2,
 'comments_at_selection': 1802,
 'title': "CFP Rankings Reaction...WHAT Are We Doing? - Josh Pate's College Football Show"}

### Reply handles

When someone replies to a reply, YouTube puts the other person's handle at the start of the text, often with **no space**: `@Bballeruk11cope` is the reply "cope". A regex can't tell where such a handle ends. So each comment's start is matched against the set of real commenter handles (`authorDisplayName`, longest match first), and the remaining text is stored as `text_body`.

The handles are used only for this match and are **not** saved in the table.

In [4]:
import re

INVISIBLE = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")

known_handles = set()   # set of lowercase '@handle' strings, used only for matching
for d in raw_files:
    for page in d["comment_threads_pages"]:
        for thread in page["items"]:
            known_handles.add(thread["snippet"]["topLevelComment"]["snippet"]["authorDisplayName"].lower())
            for reply in thread.get("replies", {}).get("comments", []):
                known_handles.add(reply["snippet"]["authorDisplayName"].lower())
    for pages in d["reply_pages"].values():
        for page in pages:
            for reply in page["items"]:
                known_handles.add(reply["snippet"]["authorDisplayName"].lower())

# group by the first 4 characters so each lookup only checks a few candidates, longest first
handles_by_prefix = {}
for h in sorted((h for h in known_handles if h.startswith("@")), key=len, reverse=True):
    handles_by_prefix.setdefault(h[:4], []).append(h)


def split_reply_handle(text):
    """Return (text without a leading @handle, status): 'none', 'removed' or 'unresolved'."""
    t = INVISIBLE.sub("", text).strip()
    if not t.startswith("@"):
        return t, "none"
    low = t.lower()
    for h in handles_by_prefix.get(low[:4], []):
        if low.startswith(h):
            return t[len(h):].strip(), "removed"
    m = re.match(r"@[\w.\-]+\s+", t)   # unknown handle, but followed by a space: the boundary is clear
    if m:
        return t[m.end():].strip(), "removed"
    return t, "unresolved"   # unknown handle glued to the text: left in place


print(f"{len(known_handles):,} handles")
for example in ["@Bballeruk11cope", "\u200b@chancekutac9680lol", "Roll Tide"]:
    print(repr(example), "->", split_reply_handle(example))

4,870 handles
'@Bballeruk11cope' -> ('cope', 'removed')
'\u200b@chancekutac9680lol' -> ('lol', 'removed')
'Roll Tide' -> ('Roll Tide', 'none')


### Flatten threads and replies into one list of records

A thread embeds at most 5 replies; the full reply chain is in `reply_pages` when there were more. Both sources can contain the same reply, so records are keyed by `comment_id` in a dict, which removes the overlap.

In [5]:
def make_record(snippet, comment_id, video_meta, is_reply, parent_id, reply_count):
    author = snippet.get("authorChannelId", {}).get("value")
    body, handle_status = split_reply_handle(snippet.get("textDisplay", ""))
    return {
        "comment_id": comment_id,
        "parent_id": parent_id,                 # None for top-level comments
        "is_reply": is_reply,
        "video_id": video_meta["video_id"],
        "creator": video_meta["creator"],
        "role": video_meta["role"],             # primary (paired videos) or supplement
        "period": video_meta["period"],         # before / announcement / after (by video publish date)
        "topic_pair": video_meta["topic_pair"],
        "video_title": video_meta["title"],
        "video_published_utc": video_meta["published_utc"],
        "author_channel_id": author,
        "is_creator": author == snippet.get("channelId"),   # the channel commenting on its own video
        "text": snippet.get("textDisplay", ""),
        "text_body": body,                      # text without a leading reply @handle
        "reply_handle": handle_status,          # none / removed / unresolved
        "like_count": snippet.get("likeCount", 0),
        "reply_count": reply_count,             # replies to this thread (top-level only)
        "published_utc": snippet["publishedAt"],
        "updated_utc": snippet.get("updatedAt"),
    }


records_by_id = {}   # dict: comment_id -> record (this also removes the reply overlap)
for d in raw_files:
    meta = videos[d["video"]["video_id"]]
    for page in d["comment_threads_pages"]:
        for thread in page["items"]:
            top = thread["snippet"]["topLevelComment"]
            records_by_id[top["id"]] = make_record(top["snippet"], top["id"], meta, False, None,
                                                   thread["snippet"]["totalReplyCount"])
            for reply in thread.get("replies", {}).get("comments", []):
                records_by_id[reply["id"]] = make_record(reply["snippet"], reply["id"], meta, True, thread["id"], None)
    for parent_id, pages in d["reply_pages"].items():
        for page in pages:
            for reply in page["items"]:
                records_by_id[reply["id"]] = make_record(reply["snippet"], reply["id"], meta, True, parent_id, None)

records = list(records_by_id.values())   # list of dicts
print(f"{len(records):,} comment records")
records[0]

9,561 comment records


{'comment_id': 'UgzlwD5tB4FymP1YSjN4AaABAg',
 'parent_id': None,
 'is_reply': False,
 'video_id': '0wNBdXIY7mQ',
 'creator': 'pate',
 'role': 'supplement',
 'period': 'before',
 'topic_pair': 'S_within_video_split',
 'video_title': "Tennessee BEATS Kentucky...FULL Reaction - Josh Pate's College Football Show",
 'video_published_utc': '2026-11-09T14:00:39Z',
 'author_channel_id': 'UCUI2ZH5pAt-Cq_zN93LC8kg',
 'is_creator': False,
 'text': 'This dude is player hater #1 when it comes to the Vols 😂😂',
 'text_body': 'This dude is player hater #1 when it comes to the Vols 😂😂',
 'reply_handle': 'none',
 'like_count': 1,
 'reply_count': 0,
 'published_utc': '2026-11-11T22:19:57Z',
 'updated_utc': '2026-11-11T22:19:57Z'}

## 3. pandas Series

In [6]:
# A Series of comment text, indexed by comment_id
text = pd.Series({r["comment_id"]: r["text"] for r in records}, name="text")
print(text.head(), "\n")
print("character length:")
print(text.str.len().describe().round(1))

UgzlwD5tB4FymP1YSjN4AaABAg                                                                  This dude is player hater #1 when it comes to the Vols 😂😂
UgwGcUQT0VKFad8P3eB4AaABAg                                             Kentucky will lose one of their next three, they are not going to the playoffs
Ugx0wlIyzVb9gHwsZUF4AaABAg    Somebody please tell me why alabama's rank. Number three and tennessee is ranked number eight I just don't understan...
UgyAiQaaumG73bKAzFd4AaABAg    Well Josh, I'm a real old school football player. And per your YT narrative you said, the score looked closer than i...
UgxKQoNGr-i039ycUfp4AaABAg    Brandon brings another aspect to the game with his legs that teams have to prepare for now. Watch Talkin Vawls film ...
Name: text, dtype: object 

character length:
count    9561.0
mean      157.0
std       208.7
min         1.0
25%        51.0
50%        99.0
75%       187.0
max      6660.0
Name: text, dtype: float64


In [7]:
# A Series of comment counts per video, built from a Counter
counts = pd.Series(Counter(r["video_id"] for r in records), name="comments").sort_values(ascending=False)
counts.index = [f'{videos[v]["creator"]:10} {videos[v]["title"][:55]}' for v in counts.index]
counts

pate       CFP Rankings Reaction...WHAT Are We Doing? - Josh Pate'    2072
pate       CFP Rankings Reaction...WHAT Are We Doing? - Josh Pate'    1802
pate       AP Poll Week 11 Reaction...MOST Overrated & Underrated     1087
pate       CFP & National Championship Predictions...UPDATED - Jos     905
crain_cone FIRST College Football Playoff Rankings Reveal LIVE REA     511
pate       Notre Dame BEATS Miami...FULL Reaction - Josh Pate's Co     489
crain_cone 24-TEAM College Football Playoff CONFIRMED | Is This GO     464
pate       Why The 24-Team CFP Is A Massive Mistake - Josh Pate's      395
pate       Week 11 Upset Alerts | Commissioner’s Poll Top 25 | CFP     385
crain_cone College Football Playoff Rankings Reveal LIVE REACTION      280
crain_cone Texas vs. LSU FULL GAME PREVIEW | Crain & Cone              242
crain_cone College Football Week 11 AP Poll Top-25 REACTION            225
crain_cone College Football Week 11 PREDICTIONS with Brooks Austin     157
crain_cone USC vs. Indian

## 4. pandas DataFrame: the comment table

In [8]:
df = pd.DataFrame(records)
df["published_utc"] = pd.to_datetime(df["published_utc"], utc=True)
df["updated_utc"] = pd.to_datetime(df["updated_utc"], utc=True)
df["video_published_utc"] = pd.to_datetime(df["video_published_utc"], utc=True)

# Time columns used in the event study
df["hours_since_upload"] = (df["published_utc"] - df["video_published_utc"]).dt.total_seconds() / 3600
df["hours_vs_announce"] = (df["published_utc"] - config.ANNOUNCE_UTC).dt.total_seconds() / 3600
df["after_announce"] = df["published_utc"] >= config.ANNOUNCE_UTC
df["after_announce_alt"] = df["published_utc"] >= config.ANNOUNCE_UTC_ALT   # sensitivity check (10:30 CT)
df["edited"] = df["updated_utc"] > df["published_utc"]

# Self-reported channel country (inferred location is done later from fan-base mentions)
df["country"] = df["author_channel_id"].map(country_by_channel)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9561 entries, 0 to 9560
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   comment_id           9561 non-null   object             
 1   parent_id            4720 non-null   object             
 2   is_reply             9561 non-null   bool               
 3   video_id             9561 non-null   object             
 4   creator              9561 non-null   object             
 5   role                 9561 non-null   object             
 6   period               9561 non-null   object             
 7   topic_pair           9561 non-null   object             
 8   video_title          9561 non-null   object             
 9   video_published_utc  9561 non-null   datetime64[ns, UTC]
 10  author_channel_id    9561 non-null   object             
 11  is_creator           9561 non-null   bool               
 12  text                

In [9]:
df.head()

,comment_id,parent_id,is_reply,video_id,creator,role,period,topic_pair,video_title,video_published_utc,...,like_count,reply_count,published_utc,updated_utc,hours_since_upload,hours_vs_announce,after_announce,after_announce_alt,edited,country
0,UgzlwD5tB4FymP1YSjN4AaABAg,None,False,0wNBdXIY7mQ,pate,supplement,before,S_within_video_split,Tennessee BEATS Kentucky...FULL Reaction - Josh Pate's College Football Show,2026-11-09 14:00:39+00:00,...,1,0.0,2026-11-11 22:19:57+00:00,2026-11-11 22:19:57+00:00,56.321667,54.832500,True,True,False,None
1,UgwGcUQT0VKFad8P3eB4AaABAg,None,False,0wNBdXIY7mQ,pate,supplement,before,S_within_video_split,Tennessee BEATS Kentucky...FULL Reaction - Josh Pate's College Football Show,2026-11-09 14:00:39+00:00,...,0,0.0,2026-11-11 15:31:41+00:00,2026-11-11 15:31:41+00:00,49.517222,48.028056,True,True,False,None
2,Ugx0wlIyzVb9gHwsZUF4AaABAg,None,False,0wNBdXIY7mQ,pate,supplement,before,S_within_video_split,Tennessee BEATS Kentucky...FULL Reaction - Josh Pate's College Football Show,2026-11-09 14:00:39+00:00,...,0,0.0,2026-11-11 15:23:51+00:00,2026-11-11 15:23:51+00:00,49.386667,47.897500,True,True,False,None
3,UgyAiQaaumG73bKAzFd4AaABAg,None,False,0wNBdXIY7mQ,pate,supplement,before,S_within_video_split,Tennessee BEATS Kentucky...FULL Reaction - Josh Pate's College Football Show,2026-11-09 14:00:39+00:00,...,0,0.0,2026-11-11 00:36:50+00:00,2026-11-11 00:36:50+00:00,34.603056,33.113889,True,True,False,None
4,UgxKQoNGr-i039ycUfp4AaABAg,None,False,0wNBdXIY7mQ,pate,supplement,before,S_within_video_split,Tennessee BEATS Kentucky...FULL Reaction - Josh Pate's College Football Show,2026-11-09 14:00:39+00:00,...,0,0.0,2026-11-10 22:45:42+00:00,2026-11-10 22:45:42+00:00,32.750833,31.261667,True,True,False,None


### Checks against YouTube's own counts

`comments_at_selection` is the `commentCount` YouTube reported when the videos were chosen. Small gaps are expected: the count includes replies that were deleted or held for review, which the API does not return.

In [10]:
check = (df.groupby("video_id").size().rename("collected").to_frame()
         .join(pd.DataFrame(videos.values()).set_index("video_id")[["creator", "role", "period", "comments_at_selection"]]))
check["comments_at_selection"] = check["comments_at_selection"].astype(int)
check["gap"] = check["comments_at_selection"] - check["collected"]
print("duplicate comment_ids:", df["comment_id"].duplicated().sum())
check.sort_values(["role", "creator", "period"])

duplicate comment_ids: 0


,collected,creator,role,period,comments_at_selection,gap
video_id,,,,,,
iGYPNyDioi8,280,crain_cone,primary,after,280,0
sISy77HQw2o,157,crain_cone,primary,after,157,0
cFgBGGl2pQg,464,crain_cone,primary,announcement,464,0
VC4NmThPfkc,511,crain_cone,primary,before,511,0
rAqrsZlhTkc,225,crain_cone,primary,before,225,0
njO_8Yg7HKk,385,pate,primary,after,385,0
vfmgUnYEL1M,2072,pate,primary,after,2072,0
_LpZ2hHLOCQ,395,pate,primary,announcement,395,0
CRR3qVcpRDY,1087,pate,primary,before,1087,0


In [11]:
summary = pd.Series({
    "comments": len(df),
    "top-level": int((~df["is_reply"]).sum()),
    "replies": int(df["is_reply"].sum()),
    "unique commenters": df["author_channel_id"].nunique(),
    "creator's own comments": int(df["is_creator"].sum()),
    "reply handle removed": int((df["reply_handle"] == "removed").sum()),
    "reply handle unresolved": int((df["reply_handle"] == "unresolved").sum()),
    "country filled (%)": round(df["country"].notna().mean() * 100, 1),
})
summary

comments                   9561.0
top-level                  4841.0
replies                    4720.0
unique commenters          4870.0
creator's own comments       20.0
reply handle removed       2187.0
reply handle unresolved      20.0
country filled (%)            5.3
dtype: float64

## 5. Save the raw dataset

In [12]:
out = config.DATA_DIR / "comments_raw.csv"
df.to_csv(out, index=False)
print(f"saved {out.relative_to(ROOT)}: {df.shape[0]:,} rows x {df.shape[1]} columns")

saved data/comments_raw.csv: 9,561 rows x 25 columns
